# 08 - M0 full-image validation matching

Run All on the same Colab GPU runtime after notebook 07. After runtime deletion, restore images with notebook 02 and NBIS with notebook 04 first. Run only one copy of this notebook at a time.

This evaluates the frozen best M0 checkpoint without retraining. All locked validation probes (impressions 2-8) are evaluated against unchanged original impression-1 galleries. No FVC2004, test_synthetic, new split, image-quality filtering or new image-pair selection. References are original impressions, not guaranteed clean.

Conditions: identity plus dry_like/wet_like/generic at mild/moderate/severe, using the exact frozen simulator and seed from notebook 06. Each condition compares input versus M0. Synthetic conditions are not verified real dry/wet labels. Images are normalized to 500 dpi, processed individually at their full native normalized size in FP32, then rounded to lossless grayscale 8-bit PNG before MINDTCT. No tiled inference, resizing to training patches or enhanced galleries.

The first M0 run barely changed patch MAE. Therefore report full-image float MAE, post-quantization MAE, changed-pixel fraction and byte-identical image fraction. Quantization can erase small model changes. Original-image changes and matching harm are explicitly measured; do not assume any original image is actually high quality.

NBIS uses the frozen notebook-05 pair lists and settings. Primary diagnostic: use each database's original-input validation threshold for EVERY condition and method. Also report per-condition/per-method thresholds calibrated to empirical conditional validation FMR <= 1%; these are diagnostic condition-aware calibrations, not deployable moisture-aware thresholds or independent test results. Include conditional FMR/FNMR, attempt FAR/FRR, low-minutiae rejections, approximate EER and byte-distinct sensitivity at the same respective primary threshold.

This is larger than a preview: 20 probe variants plus fixed galleries. MINDTCT/BOZORTH3 run on CPU even with a GPU kernel. Each completed variant is persisted with checksums and resumed by Run All. Partial variant extraction is cached locally; runtime deletion repeats only unfinished variants. Hash-verified PNGs, templates and logs are archived per variant. Do not edit frozen settings to speed up or resume a run.

Outputs: MyDrive/FingerprintResearch/evaluations/m0_validation_v1/<configuration-hash>/. Completion marker: M0 VALIDATION MATCHING COMPLETE. This remains exploratory validation on data already used for model selection, not final generalization evidence.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import json
import math
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel for full-image M0 inference.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT = NBIS_ROOT / 'install' / 'bin' / 'mindtct'
BOZORTH3 = NBIS_ROOT / 'install' / 'bin' / 'bozorth3'

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

print('GPU:', torch.cuda.get_device_name(0))
print('NBIS matching uses CPU; low GPU utilization during extraction is expected.')

In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
for name, expected in split_protocol['file_sha256'].items():
    if file_digest(SPLIT_DIR / name) != expected:
        raise RuntimeError(f'Frozen split changed: {name}')
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
validation = locked[locked['split'].eq('validation')].sort_values('relative_path').reset_index(drop=True)
if validation.empty or set(validation['dataset']) != {'FVC2000', 'FVC2002'}:
    raise RuntimeError('Expected source-only validation data.')
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
train_config = json.loads((EXPERIMENT_DIR / 'configuration.json').read_text(encoding='utf-8'))
train_completion = json.loads((EXPERIMENT_DIR / 'completion.json').read_text(encoding='utf-8'))
best = json.loads((EXPERIMENT_DIR / 'best.json').read_text(encoding='utf-8'))
split_sha = file_digest(SPLIT_DIR / 'protocol.json')
if any(config['split_protocol_sha256'] != split_sha for config in (pair_config, sim_config, train_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (file_digest(PAIR_DIR / 'pairs.csv') != pair_config['pair_csv_sha256']
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or train_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')):
    raise RuntimeError('Frozen pairs/simulator provenance mismatch.')
if (file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or train_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Simulator module mismatch.')
for name, expected in train_config['source_file_sha256'].items():
    if file_digest(EXPERIMENT_DIR / name) != expected:
        raise RuntimeError(f'Frozen training source changed: {name}')
train_sha = file_digest(EXPERIMENT_DIR / 'configuration.json')
if (train_completion['status'] != 'm0_training_complete'
        or train_completion['epochs_completed'] != train_config['epochs']
        or train_completion['configuration_sha256'] != train_sha
        or train_completion['best_checkpoint'] != best
        or best['configuration_sha256'] != train_sha):
    raise RuntimeError('Training is incomplete or best checkpoint provenance differs.')
checkpoint_path = EXPERIMENT_DIR / best['file']
if file_digest(checkpoint_path) != best['sha256']:
    raise RuntimeError('Best checkpoint checksum mismatch.')
if (str(torch.__version__) != train_config['torch_version']
        or str(torch.version.cuda) != train_config['torch_cuda']
        or Image.__version__ != train_config['pillow_version']
        or Image.__version__ != pair_config['pillow_version']
        or Image.__version__ != sim_config['pillow_version']
        or np.__version__ != sim_config['numpy_version']):
    raise RuntimeError('Runtime library versions differ from the frozen experiment.')
if not (NBIS_ROOT / 'verified_build.json').is_file():
    raise FileNotFoundError('NBIS runtime installation missing. Run notebook 04 first.')
build = json.loads((NBIS_ROOT / 'verified_build.json').read_text(encoding='utf-8'))
if (build['signature']['source_sha256'] != pair_config['nbis_source_sha256']
        or pair_config['nbis_source_sha256'] != '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
        or pair_config['mindtct_flags'] != []
        or pair_config['bozorth3_flags'] != ['-A', 'minminutiae=10']):
    raise RuntimeError('NBIS source/settings differ from the locked protocol.')
for binary in (MINDTCT, BOZORTH3):
    if not binary.is_file() or file_digest(binary) != build['binary_sha256'][binary.name]:
        raise RuntimeError(f'NBIS binary missing/modified: {binary}')
    subprocess.run([str(binary), '-version'], capture_output=True, text=True, check=True)
pairs = pd.read_csv(PAIR_DIR / 'pairs.csv', keep_default_na=False)
if pairs['duplicate_sensitivity_eligible'].dtype != bool:
    raise RuntimeError('Unexpected duplicate-sensitivity boolean encoding.')
if pairs['pair_id'].duplicated().any() or validation['relative_path'].duplicated().any():
    raise RuntimeError('Duplicate validation image/pair keys.')
probe_records = validation[validation['impression_id'].ne(1)].to_dict('records')
gallery_records = validation[validation['impression_id'].eq(1)].to_dict('records')
if (set(pairs['probe_path']) != {row['relative_path'] for row in probe_records}
        or set(pairs['gallery_path']) != {row['relative_path'] for row in gallery_records}):
    raise RuntimeError('Locked pairs do not cover the expected validation probes/galleries.')
for index, record in enumerate(validation.to_dict('records'), start=1):
    source = DATA_ROOT / record['relative_path']
    if not source.is_file() or file_digest(source) != record['sha256']:
        raise RuntimeError(f'Source missing/changed: {source}. Restore with notebook 02.')
training = load_module('m0_inference_v1', EXPERIMENT_DIR / 'restoration_baseline.py')
simulator = load_module('m0_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
model = training.RestorationUNet(base=train_config['base_channels']).to(DEVICE)
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
if checkpoint['configuration_sha256'] != train_sha or checkpoint['epoch'] != best['epoch']:
    raise RuntimeError('Checkpoint payload mismatch.')
model.load_state_dict(checkpoint['model'], strict=True)
model.eval()
del checkpoint
CONDITIONS = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Best epoch:', best['epoch'] + 1, '| Probes:', len(probe_records), '| Galleries:', len(gallery_records))
print('Locked pairs per variant:', len(pairs), '| Probe variants:', len(CONDITIONS) * 2)

In [ ]:
def quantize(image):
    if image.ndim != 2 or not np.isfinite(image).all() or image.min() < 0 or image.max() > 1:
        raise ValueError('Expected finite 2D grayscale image in [0,1].')
    return np.rint(image * 255).astype(np.uint8)

@torch.inference_mode()
def restore_image(image):
    tensor = torch.from_numpy(np.array(image, dtype=np.float32, copy=True))[None, None].to(DEVICE)
    output = model(tensor).float().cpu()[0, 0].numpy()
    if output.shape != image.shape or not np.isfinite(output).all() or output.min() < 0 or output.max() > 1:
        raise RuntimeError('M0 output shape/range/finiteness failure.')
    return output

def condition_input(record, family, severity):
    dpi = pair_config['source_dpi'][f"{record['dataset']}_{record['db']}"]
    reference = training.normalized_image(str(DATA_ROOT / record['relative_path']), dpi)
    if family == 'identity':
        return reference, reference.copy()
    seed = simulator.case_seed(record['relative_path'], family, severity, sim_config['seed'])
    degraded, _ = simulator.degrade(reference, family, severity, seed)
    return reference, degraded

def read_xyt(path):
    points = []
    for line in path.read_text(encoding='ascii').splitlines():
        if line.strip():
            values = [int(value) for value in line.split()]
            if len(values) != 4:
                raise RuntimeError(f'Malformed MINDTCT row: {path}: {line}')
            points.append(values)
    return points

def extract_png(image):
    png_bytes = image.tobytes()
    shape_bytes = json_bytes(list(image.shape))
    key = hashlib.sha256(shape_bytes + png_bytes).hexdigest()
    root = TEMPLATE_DIR / key
    png, xyt, metadata = root.with_suffix('.png'), root.with_suffix('.xyt'), root.with_suffix('.json')
    if metadata.is_file():
        cached = json.loads(metadata.read_text(encoding='utf-8'))
        if (cached['pixel_key'] != key or file_digest(png) != cached['png_sha256']
                or file_digest(xyt) != cached['template_sha256'] or len(read_xyt(xyt)) != cached['minutiae']
                or cached['status'] != ('ok' if cached['minutiae'] >= 10 else 'insufficient_minutiae')):
            raise RuntimeError(f'Cached template integrity failure: {metadata}')
        return cached
    Image.fromarray(image).save(png, dpi=(500, 500))
    result = subprocess.run([str(MINDTCT), str(png), str(root)], capture_output=True, text=True, timeout=120)
    root.with_suffix('.log').write_text(result.stdout + '\nSTDERR:\n' + result.stderr, encoding='utf-8')
    if result.returncode or not xyt.is_file():
        raise RuntimeError(f'MINDTCT failed for {png}; see {root.with_suffix(".log")}')
    count = len(read_xyt(xyt))
    cached = {'pixel_key': key, 'png_sha256': file_digest(png), 'template_sha256': file_digest(xyt),
              'minutiae': count, 'status': 'ok' if count >= 10 else 'insufficient_minutiae',
              'xyt_path': str(xyt), 'width': image.shape[1], 'height': image.shape[0]}
    write_verified(metadata, json_bytes(cached))
    return cached

def pixel_metrics(reference, degraded, output):
    before, after, target = quantize(degraded), quantize(output), quantize(reference)
    return {'float_input_mae': float(np.abs(degraded - reference).mean()),
            'float_output_mae': float(np.abs(output - reference).mean()),
            'float_change_mae': float(np.abs(output - degraded).mean()),
            'png_input_mae': float(np.abs(before.astype(float) - target).mean() / 255),
            'png_output_mae': float(np.abs(after.astype(float) - target).mean() / 255),
            'changed_pixel_fraction': float(np.mean(before != after)),
            'png_identical_to_input': bool(np.array_equal(before, after))}

def match_pairs(probes, galleries, work):
    probe_map = probes.set_index('relative_path')
    gallery_map = galleries.set_index('relative_path')
    scored = pairs.copy()
    scored['probe_status'] = scored['probe_path'].map(probe_map['status'])
    scored['gallery_status'] = scored['gallery_path'].map(gallery_map['status'])
    if scored[['probe_status', 'gallery_status']].isna().any().any():
        raise RuntimeError('Missing extraction for a locked pair.')
    scored['matchable'] = scored['probe_status'].eq('ok') & scored['gallery_status'].eq('ok')
    scored['score'] = np.nan
    indices = scored.index[scored['matchable']].tolist()
    for offset in range(0, len(indices), 256):
        selected = indices[offset:offset + 256]
        lines = []
        for row in scored.loc[selected].to_dict('records'):
            lines.extend([probe_map.loc[row['probe_path'], 'xyt_path'], gallery_map.loc[row['gallery_path'], 'xyt_path']])
        mates = work / 'mates.lis'
        mates.write_text('\n'.join(lines) + '\n', encoding='utf-8')
        result = subprocess.run([str(BOZORTH3), '-A', 'minminutiae=10', '-M', str(mates)],
                                capture_output=True, text=True, timeout=600)
        output = result.stdout.splitlines()
        log = work / f'match_{offset // 256:04d}.log'
        log.write_text(result.stdout + '\nSTDERR:\n' + result.stderr, encoding='utf-8')
        if result.returncode or len(output) != len(selected) or any(re.fullmatch(r'\s*\d+\s*', line) is None for line in output):
            raise RuntimeError(f'BOZORTH3 exit/count/format failure; see {log}')
        scored.loc[selected, 'score'] = [int(line.strip()) for line in output]
    if scored.loc[scored['matchable'], 'score'].isna().any():
        raise RuntimeError('Matchable pair has no score.')
    return scored

def archive_templates(path, rows, work):
    local_zip = work / 'templates_and_logs.zip'
    roots = sorted({str(Path(row['xyt_path']).with_suffix('')) for row in rows})
    with zipfile.ZipFile(local_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for root in roots:
            for suffix in ('.png', '.xyt', '.json', '.log'):
                item = Path(root).with_suffix(suffix)
                archive.write(item, arcname='templates/' + item.name)
        for log in sorted(work.glob('match_*.log')):
            archive.write(log, arcname='matching/' + log.name)
    expected = file_digest(local_zip)
    pending = path.with_suffix('.zip.part')
    shutil.copyfile(local_zip, pending)
    if file_digest(pending) != expected:
        raise RuntimeError(f'Archive Drive copy failed: {path}')
    pending.replace(path)

def load_unit(directory):
    marker = directory / 'completion.json'
    if not marker.is_file():
        return None
    saved = json.loads(marker.read_text(encoding='utf-8'))
    required = {'scores.csv', 'extraction.csv', 'templates_and_logs.zip'}
    if (saved['configuration_sha256'] != evaluation_sha or saved['variant'] != directory.name
            or set(saved['file_sha256']) != required):
        raise RuntimeError(f'Completed variant configuration/files differ: {directory}')
    for name, expected in saved['file_sha256'].items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Completed variant integrity failure: {directory / name}')
    scored = pd.read_csv(directory / 'scores.csv')
    extraction = pd.read_csv(directory / 'extraction.csv')
    if (scored['pair_id'].tolist() != pairs['pair_id'].tolist()
            or set(extraction['relative_path']) != {row['relative_path'] for row in probe_records}
            or len(extraction) != len(probe_records)):
        raise RuntimeError(f'Completed variant coverage differs: {directory}')
    if scored['matchable'].dtype != bool or scored['duplicate_sensitivity_eligible'].dtype != bool:
        raise RuntimeError('Saved score boolean columns have unexpected types.')
    valid_scores = scored.loc[scored['matchable'], 'score']
    if (not np.isfinite(valid_scores).all() or (valid_scores < 0).any()
            or not np.equal(valid_scores, np.floor(valid_scores)).all()
            or scored.loc[~scored['matchable'], 'score'].notna().any()):
        raise RuntimeError('Saved variant contains invalid scores/failure encoding.')
    return scored, extraction

In [ ]:
# Same score, rejection and threshold conventions as notebook 05.
def calibrate_threshold(impostor_scores, target=0.01):
    values = np.asarray(impostor_scores, dtype=float)
    if values.size == 0 or not np.isfinite(values).all():
        raise RuntimeError('Cannot calibrate without finite, matchable impostor scores.')
    if not np.equal(values, np.floor(values)).all() or (values < 0).any() or not 0 <= target < 1:
        raise ValueError('Expected nonnegative integer similarity scores and target in [0,1).')
    allowed = math.floor(target * len(values) + 1e-12)
    return int(np.sort(values)[::-1][allowed]) + 1

def conditional_curve(frame):
    valid = frame[frame['matchable']]
    genuine = np.sort(valid.loc[valid['label'].eq(1), 'score'].to_numpy(dtype=float))
    impostor = np.sort(valid.loc[valid['label'].eq(0), 'score'].to_numpy(dtype=float))
    if not len(genuine) or not len(impostor):
        raise RuntimeError('Conditional metrics need matchable genuine and impostor pairs.')
    thresholds = np.unique(np.concatenate(([0.0], genuine + 1, impostor + 1)))
    return pd.DataFrame({'threshold': thresholds,
        'conditional_fmr': (len(impostor) - np.searchsorted(impostor, thresholds, side='left')) / len(impostor),
        'conditional_fnmr': np.searchsorted(genuine, thresholds, side='left') / len(genuine)})

def metrics_at_threshold(frame, threshold):
    if frame.empty or not set(frame['label']).issubset({0, 1}):
        raise ValueError('Expected nonempty binary-labelled pair table.')
    if frame.loc[frame['matchable'], 'score'].isna().any():
        raise RuntimeError('Matchable pair has no score.')
    accepted = frame['matchable'] & frame['score'].ge(threshold)
    genuine = frame['label'].eq(1)
    impostor = ~genuine
    valid_genuine = genuine & frame['matchable']
    valid_impostor = impostor & frame['matchable']
    if not genuine.any() or not impostor.any():
        raise RuntimeError('Both genuine and impostor attempts are required.')
    return {'threshold': int(threshold), 'genuine_attempts': int(genuine.sum()),
        'impostor_attempts': int(impostor.sum()),
        'unmatchable_genuine': int((genuine & ~frame['matchable']).sum()),
        'unmatchable_impostor': int((impostor & ~frame['matchable']).sum()),
        'attempt_far': float(accepted[impostor].mean()), 'attempt_frr': float((~accepted[genuine]).mean()),
        'conditional_fmr': float(accepted[valid_impostor].mean()) if valid_impostor.any() else None,
        'conditional_fnmr': float((~accepted[valid_genuine]).mean()) if valid_genuine.any() else None}

def comparison_rows(before, after, fixed_thresholds, condition):
    left = before[before['label'].eq(1)].copy()
    right = after[after['label'].eq(1)][['pair_id', 'matchable', 'score']]
    joined = left.merge(right, on='pair_id', suffixes=('_input', '_m0'), validate='one_to_one', how='outer', indicator=True)
    if not joined['_merge'].eq('both').all():
        raise RuntimeError('Genuine comparison pair alignment failed.')
    rows = []
    for (dataset, db), group in joined.groupby(['dataset', 'db'], sort=True):
        threshold = fixed_thresholds[(dataset, db)]
        accept_before = group['matchable_input'] & group['score_input'].ge(threshold)
        accept_after = group['matchable_m0'] & group['score_m0'].ge(threshold)
        comparable = group['matchable_input'] & group['matchable_m0']
        delta = group.loc[comparable, 'score_m0'] - group.loc[comparable, 'score_input']
        rows.append({'condition': condition, 'dataset': dataset, 'db': db, 'genuine_attempts': len(group),
            'comparable_scores': int(comparable.sum()), 'median_score_delta_comparable': float(delta.median()) if len(delta) else None,
            'acceptance_recoveries': int((~accept_before & accept_after).sum()),
            'acceptance_harms': int((accept_before & ~accept_after).sum()),
            'input_attempt_frr': float((~accept_before).mean()), 'm0_attempt_frr': float((~accept_after).mean())})
    return rows

In [ ]:
import inspect
helper_functions = (quantize, restore_image, condition_input, read_xyt, extract_png, pixel_metrics,
    match_pairs, archive_templates, load_unit, calibrate_threshold, conditional_curve, metrics_at_threshold, comparison_rows)
helper_source = '\n\n'.join(inspect.getsource(function) for function in helper_functions)
evaluation_config = {
    'version': 'm0_validation_v1', 'training_configuration_sha256': train_sha,
    'best_checkpoint': best, 'pair_configuration_sha256': file_digest(PAIR_DIR / 'configuration.json'),
    'pair_csv_sha256': file_digest(PAIR_DIR / 'pairs.csv'),
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'simulator_module_sha256': sim_config['module_sha256'],
    'training_module_sha256': file_digest(EXPERIMENT_DIR / 'restoration_baseline.py'),
    'evaluation_helpers_sha256': hashlib.sha256(helper_source.encode('utf-8')).hexdigest(),
    'nbis_binary_sha256': build['binary_sha256'], 'conditions': CONDITIONS,
    'degradation_seed': sim_config['seed'], 'gallery': 'unchanged original 500dpi PNG',
    'inference': 'one full native normalized image, FP32, no AMP, no padding, no tiles',
    'quantization': 'numpy.rint(image * 255) to uint8; lossless PNG at 500dpi',
    'primary_thresholds': 'original identity/input conditional validation FMR <= 0.01 per database; fixed across all variants',
    'secondary_thresholds': 'diagnostic condition/method-specific conditional validation FMR <= 0.01 per database',
    'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
    'device_type': DEVICE.type,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'gpu_compute_capability': list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None,
    'tf32_enabled': False,
    'pillow_version': Image.__version__, 'numpy_version': np.__version__,
}
evaluation_data = json_bytes(evaluation_config)
evaluation_sha = hashlib.sha256(evaluation_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'evaluations' / 'm0_validation_v1' / evaluation_sha[:20]
CACHE_DIR = LOCAL_ROOT / 'm0_validation_v1' / evaluation_sha[:20]
TEMPLATE_DIR = CACHE_DIR / 'templates'
for directory in (REPORT_DIR, CACHE_DIR, TEMPLATE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
for name, data in {'configuration.json': evaluation_data, 'evaluation_helpers.py.txt': helper_source.encode('utf-8')}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen evaluation changed: {path}')
    if not path.exists():
        write_verified(path, data)
print('Frozen evaluation:', evaluation_sha)
print('Reports/resume directory:', REPORT_DIR)

In [ ]:
expected_units = [f'{family}_{severity}_{method}' for family, severity in CONDITIONS for method in ('input', 'm0')]
pending_units = [name for name in expected_units if load_unit(REPORT_DIR / name) is None]
galleries = None
if pending_units:
    gallery_rows = []
    for record in gallery_records:
        reference, _ = condition_input(record, 'identity', 'none')
        template = extract_png(quantize(reference))
        gallery_rows.append({**record, **template})
    galleries = pd.DataFrame(gallery_rows)
    write_csv(REPORT_DIR / 'gallery_extraction.csv', galleries)
    print('Fixed original galleries prepared:', len(galleries), flush=True)
for family, severity in CONDITIONS:
    for method in ('input', 'm0'):
        unit = f'{family}_{severity}_{method}'
        directory, work = REPORT_DIR / unit, CACHE_DIR / unit
        directory.mkdir(parents=True, exist_ok=True)
        work.mkdir(parents=True, exist_ok=True)
        if load_unit(directory) is not None:
            print('Resumed completed variant:', unit, flush=True)
            continue
        extraction_rows = []
        for index, record in enumerate(probe_records, start=1):
            reference, degraded = condition_input(record, family, severity)
            output = degraded if method == 'input' else restore_image(degraded)
            template = extract_png(quantize(output))
            extraction_rows.append({**record, **template, **pixel_metrics(reference, degraded, output),
                                    'condition': f'{family}/{severity}', 'method': method})
            if index % 100 == 0 or index == len(probe_records):
                write_csv(directory / 'extraction_progress.csv', pd.DataFrame(extraction_rows))
                print(f'{unit}: extracted/cached {index}/{len(probe_records)} probes', flush=True)
        extraction = pd.DataFrame(extraction_rows)
        scored = match_pairs(extraction, galleries, work)
        write_csv(directory / 'scores.csv', scored)
        write_csv(directory / 'extraction.csv', extraction)
        archive_templates(directory / 'templates_and_logs.zip', extraction_rows + galleries.to_dict('records'), work)
        marker = {'configuration_sha256': evaluation_sha, 'variant': unit,
                  'file_sha256': {name: file_digest(directory / name)
                                 for name in ('scores.csv', 'extraction.csv', 'templates_and_logs.zip')}}
        write_verified(directory / 'completion.json', json_bytes(marker))
        print('Saved variant:', unit, '| rejected comparisons:', int((~scored['matchable']).sum()), flush=True)
print('All 20 probe variants persisted and checksum-verified on resume.')

In [ ]:
original, _ = load_unit(REPORT_DIR / 'identity_none_input')
fixed_thresholds = {}
for key, group in original.groupby(['dataset', 'db'], sort=True):
    fixed_thresholds[key] = calibrate_threshold(group.loc[group['matchable'] & group['label'].eq(0), 'score'])
metric_rows, pixel_rows, paired_rows, threshold_rows, curve_frames = [], [], [], [], []
for family, severity in CONDITIONS:
    condition = f'{family}/{severity}'
    condition_scores = {}
    for method in ('input', 'm0'):
        scored, extraction = load_unit(REPORT_DIR / f'{family}_{severity}_{method}')
        condition_scores[method] = scored
        for (dataset, db), images in extraction.groupby(['dataset', 'db'], sort=True):
            pixel_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'probe_images': len(images), 'insufficient_minutiae': int(images['status'].ne('ok').sum()),
                'image_failure_rate': float(images['status'].ne('ok').mean()),
                **{name: float(images[name].mean()) for name in ('float_input_mae', 'float_output_mae',
                    'float_change_mae', 'png_input_mae', 'png_output_mae', 'changed_pixel_fraction')},
                'png_identical_image_fraction': float(images['png_identical_to_input'].mean())})
        for (dataset, db), group in scored.groupby(['dataset', 'db'], sort=True):
            valid_impostors = group.loc[group['matchable'] & group['label'].eq(0), 'score']
            calibrated = calibrate_threshold(valid_impostors) if len(valid_impostors) else None
            curve_available = group.loc[group['matchable'], 'label'].nunique() == 2
            eer, gap = None, None
            if curve_available:
                curve = conditional_curve(group)
                point = curve.loc[(curve['conditional_fmr'] - curve['conditional_fnmr']).abs().idxmin()]
                eer = float((point['conditional_fmr'] + point['conditional_fnmr']) / 2)
                gap = float(abs(point['conditional_fmr'] - point['conditional_fnmr']))
                curve_frames.append(curve.assign(condition=condition, method=method, dataset=dataset, db=db))
            if calibrated is None or not curve_available:
                print(f'WARNING: conditional calibration/EER unavailable for {condition}/{method}/{dataset}/{db}; all attempts retained.')
            threshold_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                'original_fixed_threshold': fixed_thresholds[(dataset, db)], 'condition_calibrated_threshold': calibrated,
                'calibration_status': 'ok' if calibrated is not None else 'no_matchable_impostors'})
            for policy, threshold in (('original_fixed', fixed_thresholds[(dataset, db)]), ('condition_calibrated', calibrated)):
                if threshold is None:
                    continue
                for view, selected in (('all_impressions', group),
                                       ('byte_distinct_sensitivity', group[group['duplicate_sensitivity_eligible']])):
                    values = metrics_at_threshold(selected, threshold)
                    if policy == 'condition_calibrated' and view == 'all_impressions' and values['conditional_fmr'] > 0.01 + 1e-12:
                        raise RuntimeError('Condition-calibrated validation FMR exceeded 1%.')
                    metric_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                        'threshold_policy': policy, 'view': view, **values,
                        'conditional_eer_discrete_approx': eer if view == 'all_impressions' else None,
                        'eer_crossing_gap': gap if view == 'all_impressions' else None})
    paired_rows.extend(comparison_rows(condition_scores['input'], condition_scores['m0'], fixed_thresholds, condition))
metrics = pd.DataFrame(metric_rows)
pixels = pd.DataFrame(pixel_rows)
paired = pd.DataFrame(paired_rows)
thresholds = pd.DataFrame(threshold_rows)
write_csv(REPORT_DIR / 'validation_metrics.csv', metrics)
write_csv(REPORT_DIR / 'pixel_and_failure_metrics.csv', pixels)
write_csv(REPORT_DIR / 'paired_genuine_changes.csv', paired)
write_csv(REPORT_DIR / 'validation_thresholds.csv', thresholds)
write_csv(REPORT_DIR / 'conditional_curves.csv', pd.concat(curve_frames, ignore_index=True))
primary = metrics[metrics['threshold_policy'].eq('original_fixed') & metrics['view'].eq('all_impressions')]
macro = primary.groupby(['condition', 'method'], sort=True).agg(
    databases=('db', 'size'), macro_attempt_frr=('attempt_frr', 'mean'), macro_attempt_far=('attempt_far', 'mean'),
    macro_conditional_fnmr=('conditional_fnmr', 'mean'), valid_fnmr_databases=('conditional_fnmr', 'count'),
    macro_conditional_fmr=('conditional_fmr', 'mean'), valid_fmr_databases=('conditional_fmr', 'count')).reset_index()
if not macro['databases'].eq(6).all():
    raise RuntimeError('Primary macro summary must include all six databases.')
write_csv(REPORT_DIR / 'macro_fixed_threshold_metrics.csv', macro)
print('PRIMARY: equal-weight database means; original-input thresholds fixed for all variants.')
print(macro.to_string(index=False))
print('\nIdentity-image harm/recovery at original thresholds:')
print(paired[paired['condition'].eq('identity/none')].to_string(index=False))
print('\nM0 image changes after 8-bit PNG quantization:')
print(pixels[pixels['method'].eq('m0')][['condition', 'dataset', 'db', 'changed_pixel_fraction',
                                      'png_identical_image_fraction']].to_string(index=False))

In [ ]:
record = probe_records[0]
figure, axes = plt.subplots(len(CONDITIONS), 3, figsize=(9, 24))
for row, (family, severity) in enumerate(CONDITIONS):
    reference, degraded = condition_input(record, family, severity)
    restored = restore_image(degraded)
    for column, (title, image) in enumerate((('input', degraded), ('M0', restored), ('reference', reference))):
        axes[row, column].imshow(quantize(image), cmap='gray', vmin=0, vmax=255)
        axes[row, column].set_title(f'{family}/{severity} | {title}', fontsize=8)
        axes[row, column].axis('off')
figure.tight_layout()
figure.savefig(REPORT_DIR / 'full_image_preview.png', dpi=140)
plt.show()
summary_files = ['configuration.json', 'evaluation_helpers.py.txt', 'gallery_extraction.csv', 'validation_metrics.csv',
    'pixel_and_failure_metrics.csv', 'paired_genuine_changes.csv', 'validation_thresholds.csv',
    'conditional_curves.csv', 'macro_fixed_threshold_metrics.csv', 'full_image_preview.png']
unit_hashes = {name: file_digest(REPORT_DIR / name / 'completion.json') for name in expected_units}
completion = {'status': 'm0_validation_matching_complete', 'configuration_sha256': evaluation_sha,
    'best_checkpoint': best, 'probe_images_per_variant': len(probe_records),
    'pairs_per_variant': len(pairs), 'variants_completed': len(expected_units),
    'unit_completion_sha256': unit_hashes,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in summary_files},
    'external_test_evaluated': False, 'independent_test_improvement_verified': False,
    'interpretation': 'exploratory validation; already used for model selection; no real moisture labels; no significance claim'}
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
print('M0 VALIDATION MATCHING COMPLETE:', REPORT_DIR)
print('Send macro_fixed_threshold_metrics, paired_genuine_changes and pixel_and_failure_metrics for the next decision.')
print('Do not add a GAN or tune on FVC2004 based only on patch MAE.')